In [13]:
import pandas as pd
import numpy as np
import re
kyc = pd.read_csv("../data/raw/track1_kyc_records.csv")
# Info
print(kyc.shape)
kyc.isnull().sum()

(36400, 12)


user_id                0
full_name              0
pan                 1896
aadhaar             2664
date_of_birth       2944
city                   0
state                  0
monthly_income      2933
occupation             0
signup_timestamp    2910
kyc_status             0
risk_segment           0
dtype: int64

In [14]:
# Column user_id

def std_id(value, prefix='USR'):
    if pd.isnull(value):
        return None
    digits = re.sub(r'[^0-9]', '', str(value))
    return f"{prefix}{digits}" if digits else None

kyc['user_id'] = kyc['user_id'].apply(std_id)
kyc['user_id'].head(20)
# To check if it worked
# failed_rows = kyc[kyc['user_id'].isna()]
# print(failed_rows)

0     USR16112
1     USR17216
2     USR45454
3     USR46189
4     USR85256
5     USR20918
6     USR22494
7     USR34742
8     USR83137
9     USR99161
10    USR16625
11    USR65060
12    USR13678
13    USR78719
14    USR83333
15    USR53554
16    USR77711
17    USR69726
18    USR53544
19    USR57699
Name: user_id, dtype: object

In [15]:
# Column full_name
 
def clean_full_name(series: pd.Series) -> pd.Series:
    """
    Clean full_name: fix OCR-style digit substitutions, normalize
    apostrophes, strip any other stray symbols, and standardize
    casing/whitespace.
    """
    names = series.astype("string").str.strip()

    # Fix OCR-style substitutions seen in this dataset: 0->o, 1->l
    names = names.str.replace("0", "o", regex=False)
    names = names.str.replace("1", "l", regex=False)

    # Normalize curly/smart apostrophes to a plain one
    names = names.str.replace("’", "'", regex=False)
    names = names.str.replace("`", "'", regex=False)

    # Remove any character that isn't a letter, space, or apostrophe
    names = names.str.replace(r"[^A-Za-z\s']", "", regex=True)

    # Collapse extra whitespace and title-case
    names = names.str.replace(r"\s+", " ", regex=True).str.strip()
    names = names.str.title()

    return names
kyc["full_name"] = clean_full_name(kyc["full_name"])
# Finds rows containing anything that is NOT a letter (a-z, A-Z) or a space (\s)
weird_names = kyc[kyc['full_name'].str.contains(r'[^a-zA-Z\s]', regex=True, na=False)]
print(weird_names['full_name'])

300          Riya D'Alia
700         Vidhi D'Alia
1069        Fariq D'Alia
1138     Jonathan D'Alia
1526       Ishaan D'Alia
              ...       
34525        Ojas D'Alia
34626      Watika D'Alia
35538       Isaac D'Alia
35601      Upasna D'Alia
35846     Praneel D'Alia
Name: full_name, Length: 75, dtype: string


In [16]:
# Column pan

# 1. Normalize: uppercase, strip outer whitespace, remove internal spaces/hyphens
pan_clean = (
    kyc['pan'].astype('string').str.strip().str.upper().str.replace(r'[\s-]', '', regex=True)
)

# 2. Identify conditions
is_null = pan_clean.isna() | (pan_clean == '')
is_valid = pan_clean.str.fullmatch(r'[A-Z]{5}[0-9]{4}[A-Z]', na=False)
is_invalid = ~is_null & ~is_valid

# 3. Keep the cleaned value as its own column, add validity flags
# instead of overwriting the raw data
kyc['pan_clean'] = pan_clean
kyc['pan_status'] = 'VALID'
kyc.loc[is_invalid, 'pan_status'] = 'INVALID'
kyc.loc[is_null, 'pan_status'] = 'MISSING'
kyc['pan_status'].value_counts()
kyc['pan'] = kyc['pan_clean']
kyc = kyc.drop(columns=['pan_clean'])
columns = ['pan', 'pan_status']
kyc[columns].head()

,pan,pan_status
0,SEJAA8194O,VALID
1,QT0ZZ5561X,INVALID
2,QCNTL9489O,VALID
3,RRVUI4059Q,VALID
4,YGVOI9236W,VALID


In [17]:
# Column aadhar

# 1. Normalize: uppercase, strip outer whitespace, remove internal spaces/hyphens
aadhaar_clean = (
    kyc['aadhaar']
    .astype('string')
    .str.strip()
    .str.upper()
    .str.replace(r'[\s-]', '', regex=True)
)

# 2. Recover scientific-notation values that resolve to exactly 12 digits
#    (e.g. '2.34567E+11' -> '234567000000') — not fabricating, only recover when the math produces a clean 12-digit number
scientific_mask = aadhaar_clean.str.fullmatch(
    r'[0-9]+(?:\.[0-9]+)?E[+-]?[0-9]+',
    na=False
)
numeric_values = pd.to_numeric(
    aadhaar_clean.where(scientific_mask),
    errors='coerce'
)
integer_mask = (
    numeric_values.notna()
    & (numeric_values % 1 == 0)
)
recovered = numeric_values.where(integer_mask).map(
    lambda x: f'{int(x):.0f}' if pd.notna(x) else pd.NA
)

recovered_valid = recovered.str.fullmatch(r'[0-9]{12}', na=False)
aadhaar_clean.loc[recovered_valid] = recovered.loc[recovered_valid]

# 3. Identify conditions
is_null = aadhaar_clean.isna() | (aadhaar_clean == '')
is_masked = aadhaar_clean.str.fullmatch(r'X{8}[0-9]{4}', na=False)  # e.g. XXXXXXXX1234
is_valid = aadhaar_clean.str.fullmatch(r'[0-9]{12}', na=False)
is_invalid = ~is_null & ~is_valid & ~is_masked

# 4. Keep the cleaned value in its own column, flag status separately
#    — nothing gets overwritten, so invalid/missing rows stay inspectable
kyc['aadhaar_clean'] = aadhaar_clean
kyc['aadhaar_status'] = 'VALID'
kyc.loc[is_masked, 'aadhaar_status'] = 'MASKED'
kyc.loc[is_invalid, 'aadhaar_status'] = 'INVALID'
kyc.loc[is_null, 'aadhaar_status'] = 'MISSING'
kyc['aadhaar_status'].value_counts()
kyc['aadhaar'] = kyc['aadhaar_clean']
kyc = kyc.drop(columns=['aadhaar_clean'])
columns = ['aadhaar', 'aadhaar_status']
kyc[columns].head()

,aadhaar,aadhaar_status
0,715658320763,VALID
1,023412025028,VALID
2,278162999816,VALID
3,066780327731,VALID
4,176258817110,VALID


In [28]:
# Column: date_of_birth

# 1. Ensure everything is a clean string for checking
temp_dob = kyc['date_of_birth'].astype(str).str.strip()

# 2. Identify which rows are Unix timestamps
is_unix = temp_dob.str.match(r'^-?\d+$', na=False)

# 3. Convert the Unix timestamps
unix_dates = pd.to_datetime(temp_dob[is_unix].astype(float), unit='s', errors='coerce')

# 4. Convert the standard date strings (ADDED format='mixed' HERE)
standard_dates = pd.to_datetime(temp_dob[~is_unix], errors='coerce', format='mixed')

# 5. Combine them back into the main column
kyc['date_of_birth_clean'] = standard_dates.combine_first(unix_dates)

# --- NEW: Age Checks ---
# Find the exact cutoff dates for 18 and 70 years old today
eighteen_years_ago = pd.Timestamp.today() - pd.DateOffset(years=18)
seventy_years_ago = pd.Timestamp.today() - pd.DateOffset(years=70)

# Create our True/False conditions
is_missing = kyc['date_of_birth_clean'].isna()
is_underage = kyc['date_of_birth_clean'] > eighteen_years_ago  # Born AFTER 18 years ago
is_overage = kyc['date_of_birth_clean'] < seventy_years_ago    # Born BEFORE 70 years ago

# Flag everything in a single column
kyc['age_status'] = 'VALID'
kyc.loc[is_underage, 'age_status'] = 'UNDERAGE'
kyc.loc[is_overage, 'age_status'] = 'OVERAGE'
kyc.loc[is_missing, 'age_status'] = 'MISSING_DOB'

# Finally, drop the time and keep just YYYY-MM-DD
kyc['date_of_birth_clean'] = kyc['date_of_birth_clean'].dt.date
print(kyc['age_status'].value_counts())
kyc['date_of_birth'] = kyc['date_of_birth_clean']
kyc = kyc.drop(columns=['date_of_birth_clean'])
columns = ['date_of_birth', 'age_status']
kyc[columns].head()

age_status
VALID          33456
MISSING_DOB     2944
Name: count, dtype: int64


,date_of_birth,age_status
0,1967-06-04,VALID
1,NaT,MISSING_DOB
2,NaT,MISSING_DOB
3,NaT,MISSING_DOB
4,1969-02-19,VALID


In [19]:
# Column: city

kyc['city'] = kyc['city'].astype(str).str.strip().str.title()

# 2. Create a dictionary mapping the 'bad' values to the 'good' values
city_mapping = {
    'Asr': 'Amritsar',
    'Bangalore': 'Bengaluru',
    'Blr': 'Bengaluru',
    'Bombay': 'Mumbai',
    'Mumbay': 'Mumbai',
    'Calcutta': 'Kolkata',
    'Dilli': 'Delhi',
    'New Delhi': 'Delhi',
    'Hyd': 'Hyderabad',
    'Jalandar': 'Jalandhar',
    'Jpr': 'Jaipur',
    'Ldh': 'Ludhiana',
    'Lko': 'Lucknow',
    'Madras': 'Chennai',
    'Poona': 'Pune'
}

# 3. Replace the values
# This will ONLY change the ones in the dictionary, and leave everything else alone
kyc['city'] = kyc['city'].replace(city_mapping)
# Check
print("Cities:", sorted(kyc['city'].dropna().unique()))

Cities: ['Amritsar', 'Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Jalandhar', 'Kolkata', 'Lucknow', 'Ludhiana', 'Mumbai', 'Pune']


In [20]:
# Column: States 

print("States:", sorted(kyc['state'].dropna().unique()))


States: ['Delhi', 'Karnataka', 'Maharashtra', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'West Bengal']


In [21]:
# Column: monthly_income

def clean_amount(value):
    # Catches actual programmatic nulls (np.nan, None)
    if pd.isnull(value):
        return None
    
    # 1. Convert to lowercase and strip extra spaces on the edges
    v = str(value).lower().strip()
    
    # 2. THE NULL CATCHER: Intercept text-based empty values
    if v in ['not available', 'na', 'n/a', 'nan', 'null', 'missing', 'none', '-', '']:
        return None
        
    # 3. Erase symbols, spaces, commas AND currency words
    v = re.sub(r'[₹$,\s]|inr|rs\.?|rupees?', '', v)
    
    multiplier = 1
    
    # 4. Check for the value suffixes
    if v.endswith('k'):
        multiplier = 1_000
        v = v.replace('k', '')
        
    elif v.endswith('l') or v.endswith('lakh'):
        multiplier = 100_000
        v = re.sub(r'(l|lakh)$', '', v)
        
    elif v.endswith('cr') or v.endswith('crore'):
        multiplier = 10_000_000
        v = re.sub(r'(cr|crore)$', '', v)
        
    elif v.endswith('m'):
        multiplier = 1_000_000
        v = v.replace('m', '')

    # 5. Convert and multiply
    try:
        return float(v) * multiplier
    except ValueError:
        return None

# Apply the cleaning function
kyc['monthly_income_clean'] = kyc['monthly_income'].apply(clean_amount)

# Define conditions to categorize monthly income using the cleaned numeric values
conditions = [
    kyc['monthly_income_clean'].isna(),                    # Missing or null values
    kyc['monthly_income_clean'] < 0,                       # Negative values (data error or liability)
    kyc['monthly_income_clean'] == 0,                      # Zero declared income
    kyc['monthly_income_clean'] > 1_000_000                # Ultra-high income anomaly threshold (>10 Lakhs/month)
]
choices = ['MISSING', 'NEGATIVE', 'ZERO', 'HIGH_INCOME']

# Generate the single categorical status column
kyc['income_status'] = np.select(conditions, choices, default='STANDARD_INCOME')

# Finalize columns
kyc['monthly_income'] = kyc['monthly_income_clean']
kyc = kyc.drop(columns=['monthly_income_clean'])

# Check results
print(kyc['income_status'].value_counts(dropna=False))

income_status
STANDARD_INCOME    30221
MISSING             4723
NEGATIVE            1456
Name: count, dtype: int64


In [22]:
# Column: occupation
kyc['occupation'].unique()

array(['Retired', 'Freelancer', 'Farmer', 'Student', 'Salaried',
       'Self Employed', 'Unemployed', 'Gig Worker', 'Business Owner'],
      dtype=object)

In [23]:
# Column: signup_timestamp

# 1. Ensure everything is a clean string for checking
raw_signup = kyc['signup_timestamp'].astype(str).str.strip().str.lower()

# 2. Identify entries missing a time component (the Midnight Trap indicator)
is_valid_string = ~raw_signup.isin(['nan', 'none', 'nat', 'null', ''])
no_time_provided = ~raw_signup.str.contains(':', na=False)

# 3. Convert to proper datetime and strip the time component (normalize to date midnight 00:00:00)
kyc['signup_date_clean'] = pd.to_datetime(kyc['signup_timestamp'], errors='coerce', format='mixed').dt.normalize()

# 4. Calculate Account Age in Days using date-only math
kyc['account_age_days'] = (pd.Timestamp.today().normalize() - kyc['signup_date_clean']).dt.days

# 5. Combine status checks into a single categorical column
conditions = [
    kyc['signup_date_clean'].isna(),                             # Missing date entirely
    kyc['account_age_days'] < 0,                                      # Impossible future date
    kyc['account_age_days'] == 0,                                     # Brand new (last 24 hours)
    is_valid_string & no_time_provided                                # Valid date, but time was missing in raw data
]
choices = ['MISSING_DATE', 'FUTURE_DATE', 'BRAND_NEW', 'DATE_ONLY']

kyc['signup_status'] = np.select(conditions, choices, default='VALID')

# 6. Finalize columns (retaining only the date component)
kyc['signup_date'] = kyc['signup_date_clean']
kyc = kyc.drop(columns=['signup_date_clean'])

# Check the breakdown of the consolidated status
print(kyc['signup_status'].value_counts())

columns = ['signup_date', 'account_age_days', 'signup_status']
kyc[columns].head(10)

signup_status
DATE_ONLY       21634
VALID            8823
MISSING_DATE     5823
FUTURE_DATE       120
Name: count, dtype: int64


,signup_date,account_age_days,signup_status
0,2025-12-02,285.0,VALID
1,2024-01-31,956.0,DATE_ONLY
2,2026-01-30,226.0,VALID
3,2025-09-04,374.0,DATE_ONLY
4,2026-02-03,222.0,DATE_ONLY
5,2024-04-25,871.0,DATE_ONLY
6,NaT,NaN,MISSING_DATE
7,2025-07-25,415.0,DATE_ONLY
8,2024-03-04,923.0,DATE_ONLY
9,NaT,NaN,MISSING_DATE


In [24]:
# Column: kyc_status

# 1. Force to string, lowercase, and strip spaces
kyc['kyc_status'] = kyc['kyc_status'].astype(str).str.strip().str.lower()

# 2. The Fraud-Optimized Mapping Dictionary
status_mapping = {
    # The Approved Bucket
    'done': 'APPROVED',
    'verified': 'APPROVED',
    'approved': 'APPROVED',
    'kyc_done': 'APPROVED',
    'v': 'APPROVED',
    
    # The 'Waiting in Line' Bucket (Low/Unknown Risk)
    'pending': 'PENDING',
    'p': 'PENDING',
    
    # The 'Suspicious/Manual Check' Bucket (Higher Risk Flag!)
    'under review': 'UNDER_REVIEW',
    'in_progress': 'UNDER_REVIEW',
    
    # The Rejected Bucket
    'rejected': 'REJECTED',
    'reject': 'REJECTED',
    'failed': 'REJECTED',
    'r': 'REJECTED',
    
    # Missing data
    'nan': 'UNKNOWN'
}

# 3. Apply the mapping
kyc['kyc_status_clean'] = kyc['kyc_status'].replace(status_mapping)

print(kyc['kyc_status_clean'].value_counts())
kyc['kyc_status_clean'].unique()

kyc['kyc_status'] = kyc['kyc_status_clean']
kyc = kyc.drop(columns = ['kyc_status_clean'])



kyc_status_clean
APPROVED        27764
PENDING          3500
REJECTED         3169
UNDER_REVIEW     1967
Name: count, dtype: int64


In [25]:
# Column: risk_segment

risk_mapping = {
    'LOW': 'Low',
    'medium': 'Medium',
    'low': 'Low',
    'MEDIUM': 'Medium',
    'high': 'High',
    'HIGH': 'High',
    'UNKNOWN': 'Unknown',
    'unknown': 'Unknown'
}
kyc['risk_segment'] = kyc['risk_segment'].replace(risk_mapping)
# Check
print("Risks:", sorted(kyc['risk_segment'].dropna().unique()))

Risks: ['High', 'Low', 'Medium', 'Unknown']


In [26]:
kyc = kyc[[
    'user_id', 
    'full_name', 
    'pan', 
    'pan_status', 
    'aadhaar', 
    'aadhaar_status', 
    'date_of_birth', 
    'age_status', 
    'city', 
    'state', 
    'monthly_income', 
    'income_status', 
    'occupation', 
    'signup_timestamp', 
    'signup_date', 
    'account_age_days', 
    'signup_status', 
    'risk_segment', 
    'kyc_status'
]]

In [27]:
kyc.to_csv('../data/cleaned/modified_kyc_records.csv', index=False)